# Experiment 2 - Mapping Cost Criteria Comparison

This notebook compares two stage-cost designs on **the same mapping model**:

- Shannon entropy cost: `r(b) = H(b)`
- Rao entropy cost: `r_W(b) = b^T D b`

The transition model, quantization, and rollout environment are held fixed. We evaluate both learned policies under a common Monte Carlo metric: map MSE against the ground-truth landmark map.

In [ ]:
import sys
import os
import subprocess
from pathlib import Path
import inspect
import numpy as onp
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

# CRITICAL: Preload libcuda.so.1 from system path BEFORE anything else
import ctypes
try:
    ctypes.CDLL("/usr/lib64/libcuda.so.1", mode=ctypes.RTLD_GLOBAL)
    print("✓ Preloaded libcuda.so.1 from /usr/lib64")
except Exception as e:
    print(f"⚠ Could not preload libcuda.so.1: {e}")

# CuPy: ensure CUDA_PATH and LD_LIBRARY_PATH
if "CUDA_PATH" not in os.environ:
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True, executable='/bin/bash', capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            raise RuntimeError("CUDA_PATH not set. Run: module load cuda/12.2")
    except Exception as e:
        raise RuntimeError(f"Failed to load CUDA: {e}") from e

cuda_path = os.environ.get('CUDA_PATH')
if cuda_path:
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),
    ]
    current_ld = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld.split(':') if current_ld else []
    for p in cuda_lib_paths:
        if os.path.exists(p) and p not in ld_paths:
            ld_paths.insert(0, p)
    if ld_paths != (current_ld.split(':') if current_ld else []):
        os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
        print(f"✓ Updated LD_LIBRARY_PATH")
    for p in cuda_lib_paths:
        if not os.path.exists(p):
            continue
        for name in ['libcudart.so', 'libcudart.so.12', 'libnvrtc.so.12']:
            path = os.path.join(p, name)
            if os.path.exists(path):
                try:
                    ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
                    print(f"✓ Preloaded {name}")
                except Exception as e:
                    print(f"⚠ Could not preload {name}: {e}")
                break

print("\n=== Environment before CuPy ===")
print(f"CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
print("=====================================\n")

from src.utils.array_backend import np, random, is_cupy
from src.belief_quantized.belief_mdp_n_M import BeliefMDP_n_M_Mapping, BeliefMDP_n_M_Localization
from src.classes.model import SingleIntegratorModel, DoubleIntegratorModel, RangeBearingSensor
from src.classes.mapping import LandmarkMap
from src.belief_quantized.value_iteration import ValueIteration

import warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='cupy.random')

print("✓ All imports successful")

if is_cupy:
    try:
        import cupy as cp
        print("Using backend: CuPy (GPU)")
        print(f"✓ CuPy {cp.__version__}, CUDA devices: {cp.cuda.runtime.getDeviceCount()}")
    except Exception as e:
        print(f"✗ CuPy check failed: {e}")
else:
    print("Using backend: NumPy (CPU)")

def to_numpy(x):
    return x.get() if hasattr(x, 'get') else onp.asarray(x)

def to_backend(x):
    return np.asarray(x)


In [ ]:
# Quantization — pose and map grids are now independent
pose_n = 4   # robot state quantization: m_n = pose_n^2 = 16 (SingleIntegrator, 2D)
map_n  = 2   # landmark map quantization: K = map_n² = 4 cell centres, len_M = K^L = 16
obs_n = 3
action_n = 4
M = 6
beta = 0.95
epsilon = 1e-6
n_gpus = 8   # number of GPUs for parallel p_n_M computation

# Map M = W^l: W = map_n×map_n grid over [0,10]² → K=4 centres; l=2 → len_M = 16
landmark_map = LandmarkMap(
    x_min=0.0, x_max=10.0,
    y_min=0.0, y_max=10.0,
    n=map_n,
    num_landmarks=2,
)

# Noise — reduced to let the Bayesian filter converge
sigma_w = 0.1    # process noise — low for sharper T matrix (more action differentiation)
sigma_r = 0.75    # range observation noise
sigma_phi = 0.25  # bearing observation noise
cov_y = onp.diag(onp.tile([sigma_r**2, sigma_phi**2], landmark_map.num_candidate_landmarks))

# Monte Carlo setup
horizon = 20
n_trials = 100

# SingleIntegrator: 2D state (x, y), velocity control, m_n = pose_n² = 16
# max_v must exceed grid spacing (10/pose_n = 2.5) so the robot can
# actually transition between quantized cells in one step.
motion_model = SingleIntegratorModel(
    i_x=5.0, i_y=5.0, dt=1.0, max_v=4.0
)
sensor = RangeBearingSensor(r_max=6.0, epsilon=0.1, sigma_r=sigma_r, sigma_phi=sigma_phi)

obstacles = []

# Free GPU 0 memory before multi-GPU computation
if is_cupy:
    import cupy as cp
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()

print('Building BeliefMDP_n_M_Mapping...')
bmdp = BeliefMDP_n_M_Mapping(
    M=M,
    β=beta,
    n=pose_n,
    motion_model=motion_model,
    measurement_model=sensor,
    obstacles=obstacles,
    _map=landmark_map,
    sigma_w=sigma_w,
    cov_y=cov_y,
    obs_n=obs_n,
    action_n=action_n,
    j_batch_size=None,
    i_batch_size=128,   # reduced for multi-GPU (each GPU needs headroom for intermediates)
    n_gpus=n_gpus,
)

print('Built model with:')
print(f"  n (pose):        {bmdp.n}  → m_n = {bmdp.SQ.m_n} robot states")
print(f"  n_map (map):     {bmdp.n_map}  → K={bmdp.map.n_cells} cells, len_M={bmdp.len_M}")
print(f"  |Pi_n^M| cardinality: {bmdp.BQ.cardinality}")
print(f"  n_u (actions):   {bmdp.AQ.n_u}")
print(f"  obs_n:           {getattr(bmdp, 'obs_n', None)}")
print(f"  state_dim:       {bmdp.state_dim}")
print(f"  max_v:           {motion_model.max_v}  (grid spacing = {10.0/pose_n:.2f})")
print(f"  n_gpus:          {n_gpus}")

In [ ]:
# ----------------------------
# Consistency checks
# ----------------------------
mapping_sig = inspect.signature(BeliefMDP_n_M_Mapping.__init__)
loc_sig = inspect.signature(BeliefMDP_n_M_Localization.__init__)

print('BeliefMDP_n_M_Mapping.__init__:', mapping_sig)
print('BeliefMDP_n_M_Localization.__init__:', loc_sig)

required_attrs = ['Q_n', 'Y_n', 'AQ', 'SQ', 'BQ', 'p_n_M', 'c_n_M']
missing = [a for a in required_attrs if not hasattr(bmdp, a)]
if missing:
    raise RuntimeError(f'Missing required mapping attributes: {missing}')

if bmdp.Q_n is None:
    raise RuntimeError('Q_n is None; observation quantization was not initialized correctly.')

print('Structure check passed: mapping class has expected quantized-model components.')

# Build two cost tensors on top of IDENTICAL dynamics/transitions
B_codebook = to_backend(bmdp.BQ.Π_n_M)                 # (cardinality, len_M)
U_n = to_backend(bmdp.AQ.U)                            # (n_u, action_dim)
# NOTE: Remove control/effort penalty so both costs are purely information-based.
shannon = to_backend(bmdp.r_information_gain(B_codebook))  # (cardinality,)
rao = to_backend(bmdp.rao_entropy(B_codebook))             # (cardinality,)

cardinality = bmdp.BQ.cardinality
m_n = bmdp.SQ.m_n
n_u = bmdp.AQ.n_u

# Pure-information stage costs (no control penalty)
c_shannon_2d = shannon[:, np.newaxis]                           # (cardinality, n_u)
c_rao_2d = rao[:, np.newaxis]                                   # (cardinality, n_u)

c_shannon = np.broadcast_to(c_shannon_2d[:, np.newaxis, :], (cardinality, m_n, n_u)).copy()
c_rao = np.broadcast_to(c_rao_2d[:, np.newaxis, :], (cardinality, m_n, n_u)).copy()

print('Custom stage costs built with shared dynamics:')
print('  c_shannon shape:', c_shannon.shape)
print('  c_rao shape:', c_rao.shape)


In [ ]:
# ----------------------------
# Value iteration under each cost
# ----------------------------
def run_vi_with_cost(cost_tensor, label):
    old_cost = bmdp.c_n_M
    try:
        bmdp.c_n_M = cost_tensor
        vi = ValueIteration(bmdp, epsilon=epsilon)
        vi.run(verbose=True)
        out = {
            'label': label,
            'policy': to_numpy(vi.policy).astype(onp.int32),
            'V': to_numpy(vi.V),
            'iterations': vi.iteration_count,
        }
    finally:
        bmdp.c_n_M = old_cost
    return out

vi_shannon = run_vi_with_cost(c_shannon, 'shannon')
vi_rao = run_vi_with_cost(c_rao, 'rao')

print('Done: both policies computed.')
print('  Shannon iterations:', vi_shannon['iterations'])
print('  Rao iterations:', vi_rao['iterations'])


In [ ]:
# ----------------------------
# Policy & Cost Diagnostic: Are Shannon and Rao actually different?
# ----------------------------
pol_sh = vi_shannon['policy']  # (cardinality, m_n)
pol_ra = vi_rao['policy']

# (i) Are the policy arrays identical?
policies_identical = onp.array_equal(pol_sh, pol_ra)
n_differ = int(onp.sum(pol_sh != pol_ra))
n_total = pol_sh.size
pct_differ = 100.0 * n_differ / n_total

print("=" * 60)
print("POLICY COMPARISON DIAGNOSTIC")
print("=" * 60)
print(f"  Policies identical?  {policies_identical}")
print(f"  Entries that differ: {n_differ:,} / {n_total:,} ({pct_differ:.2f}%)")

# Where do they differ? Break down by pose state
if n_differ > 0:
    diff_mask = pol_sh != pol_ra
    for x_idx in range(pol_sh.shape[1]):
        n_diff_x = int(diff_mask[:, x_idx].sum())
        print(f"    x_idx={x_idx}: {n_diff_x:,} differ out of {pol_sh.shape[0]:,}")
else:
    print("  -> Policies are PERFECTLY IDENTICAL despite different cost functions.")

# (ii) Compare the cost vectors themselves
shannon_np = to_numpy(shannon)
rao_np = to_numpy(rao)

# Are the costs monotonically related? (If so, argmin is always the same)
from scipy.stats import spearmanr, kendalltau
rho_spearman, p_spearman = spearmanr(shannon_np, rao_np)
tau_kendall, p_kendall = kendalltau(shannon_np, rao_np)

print(f"\n  Cost correlation (over {cardinality:,} codebook beliefs):")
print(f"    Spearman rho = {rho_spearman:.6f}  (p = {p_spearman:.2e})")
print(f"    Kendall tau  = {tau_kendall:.6f}  (p = {p_kendall:.2e})")

# If Spearman ~ 1.0, the two costs rank beliefs identically,
# so the optimal future-value ordering is the same -> same policy.

# (iii) Check how many unique actions each policy uses
for name, pol in [("Shannon", pol_sh), ("Rao", pol_ra)]:
    unique_actions = onp.unique(pol)
    action_counts = {int(a): int((pol == a).sum()) for a in unique_actions}
    print(f"\n  {name} policy action distribution:")
    for a, cnt in sorted(action_counts.items()):
        print(f"    action {a}: {cnt:,} states ({100*cnt/n_total:.1f}%)")

# (iv) Check if V functions differ (even if policies are the same)
V_sh = vi_shannon['V']
V_ra = vi_rao['V']
# Normalize both to [0,1] for comparable scale
V_sh_norm = (V_sh - V_sh.min()) / max(V_sh.max() - V_sh.min(), 1e-15)
V_ra_norm = (V_ra - V_ra.min()) / max(V_ra.max() - V_ra.min(), 1e-15)
V_corr = float(onp.corrcoef(V_sh_norm.ravel(), V_ra_norm.ravel())[0, 1])
print(f"\n  Value function correlation (normalized): {V_corr:.6f}")

# (v) Q-value margin analysis: how close is the 2nd-best action?
# If margins are tiny, even small cost differences SHOULD flip the argmin.
# If margins are large, the transition structure dominates.
print(f"\n  Q-value margin analysis (how decisive is the argmin?):")
for label, cost_tensor in [("Shannon", c_shannon), ("Rao", c_rao)]:
    bmdp.c_n_M = cost_tensor
    vi_temp = ValueIteration(bmdp, epsilon=epsilon)
    vi_temp.run(verbose=False)
    # Recompute Q-values
    V_flat = vi_temp.V.T.reshape(-1)
    q_list = []
    for k in range(n_u):
        Ev = bmdp.p_n_M[k] @ V_flat
        q_list.append(Ev.reshape(m_n, cardinality).T)
    q_all = np.stack(q_list, axis=2)  # (cardinality, m_n, n_u)
    q_all_np = to_numpy(q_all + cost_tensor)
    
    # For each (belief, state), compute gap between best and 2nd-best action
    q_sorted = onp.sort(q_all_np, axis=2)
    margin = q_sorted[:, :, 1] - q_sorted[:, :, 0]  # 2nd_best - best (always >= 0)
    print(f"    {label}: margin mean={margin.mean():.6f}, median={onp.median(margin):.6f}, "
          f"max={margin.max():.4f}, zero-margin%={100*(margin < 1e-10).mean():.1f}%")

# Restore original cost
bmdp.c_n_M = to_backend(bmdp.c_n_M)
print("=" * 60)

In [ ]:
# ----------------------------
# Monte Carlo rollout evaluator (CPU-optimized, no GPU round-trips)
# ----------------------------
from scipy.spatial import KDTree as ScipyKDTree

# ---- Pre-convert everything to CPU numpy once ----
all_maps = to_numpy(bmdp.all_maps_3d)       # (len_M, L, 2)
Q_n_np = to_numpy(bmdp.Q_n)                 # (m_y, m_n, len_M)
Y_n_np = to_numpy(bmdp.Y_n)                 # (m_y, obs_dim)
X_n_np = to_numpy(bmdp.SQ.X_n)              # (m_n, state_dim)
U_np = to_numpy(bmdp.AQ.U)                  # (n_u, 2)
codebook_np = to_numpy(bmdp.BQ.Π_n_M)       # (cardinality, len_M)
D_map_np = to_numpy(bmdp._get_map_distance_matrix())  # (len_M, len_M)

L_lm = all_maps.shape[1]
len_M = all_maps.shape[0]
state_dim = bmdp.state_dim
cov_x = to_numpy(bmdp.cov_x)
x0 = onp.array([5.0, 5.0], dtype=float)

_dt = float(motion_model.dt)
_state_lower = to_numpy(bmdp.state_bounds[:, 0])
_state_upper = to_numpy(bmdp.state_bounds[:, 1])
_eps = float(sensor.epsilon)
_r_max = float(sensor.r_max)
_sigma_r = float(sensor.sigma_r)
_sigma_phi = float(sensor.sigma_phi)

# ---- Hash-based codebook lookup (O(1) instead of O(cardinality)) ----
_M_q = int(bmdp.BQ.M)
_N_n = int(bmdp.BQ.N_n)
_codebook_int = onp.rint(codebook_np * _M_q).astype(onp.int64)
_codebook_hash = {}
for _idx, _row in enumerate(_codebook_int):
    _codebook_hash[tuple(_row)] = _idx

def _reznik_quantize(b):
    """Inline Reznik algorithm — pure numpy."""
    k = onp.floor(_M_q * b + 0.5).astype(onp.int64)
    delta = int(k.sum()) - _M_q
    if delta != 0:
        d = k - _M_q * b
        order = onp.argsort(d)
        if delta > 0:
            k[order[-delta:]] -= 1
        else:
            k[order[:-delta]] += 1
    return k

def fast_belief_to_index(b):
    """Quantize + hash lookup — O(len_M) instead of O(cardinality)."""
    k = _reznik_quantize(b)
    idx = _codebook_hash.get(tuple(k))
    if idx is not None:
        return idx
    # Rare fallback: nearest neighbor
    q = k.astype(onp.float64) / _M_q
    return int(onp.argmin(onp.linalg.norm(codebook_np - q, axis=1)))

# ---- Fast state index via KDTree ----
_state_tree = ScipyKDTree(X_n_np)

def fast_state_index(x):
    _, idx = _state_tree.query(x)
    return int(idx)

# ---- Fast observation index (brute-force over tiny Y_n) ----
_Y_n_nan = onp.isnan(Y_n_np)

def fast_obs_index(y_flat):
    """Find nearest Y_n index. y_flat: (obs_dim,)."""
    nan_y = onp.isnan(y_flat)
    if onp.any(nan_y):
        matches = onp.all(_Y_n_nan == nan_y, axis=1)
        if onp.any(matches):
            non_nan = ~nan_y
            dists = onp.linalg.norm(Y_n_np[matches][:, non_nan] - y_flat[non_nan], axis=1)
            return int(onp.where(matches)[0][onp.argmin(dists)])
    dists = onp.linalg.norm(onp.nan_to_num(Y_n_np) - onp.nan_to_num(y_flat), axis=1)
    dists = onp.where(_Y_n_nan.any(axis=1), onp.inf, dists)
    return int(onp.argmin(dists))

# ---- Fast filter update — pure numpy log-space ----
def fast_filter(b, x_idx, y_flat):
    obs_idx = fast_obs_index(y_flat)
    log_Q = onp.log(onp.maximum(Q_n_np[obs_idx, x_idx, :], 1e-300))
    log_b = onp.log(onp.maximum(b, 1e-300))
    log_num = log_Q + log_b
    log_num -= log_num.max()           # shift for numerical stability
    b_new = onp.exp(log_num)
    s = b_new.sum()
    return b_new / s if s > 0 else onp.full(len_M, 1.0 / len_M)

# ---- Fast dynamics and sensor — pure numpy ----
def fast_step(x, u, w):
    return onp.clip(x + u * _dt + w, _state_lower, _state_upper)

def fast_observe(x, true_map, v):
    """Range-bearing obs with noise. Returns (L*2,) flat array."""
    delta = true_map - x[:2]                                   # (L, 2)
    ranges = onp.sqrt((delta**2).sum(axis=1))                  # (L,)
    bearings = onp.arctan2(delta[:, 1], delta[:, 0])           # (L,)
    visible = (ranges >= _eps) & (ranges <= _r_max)
    z = onp.full((L_lm, 2), onp.nan)
    if visible.any():
        z[visible, 0] = ranges[visible] + v[visible, 0]
        noisy_phi = bearings[visible] + v[visible, 1]
        z[visible, 1] = onp.arctan2(onp.sin(noisy_phi), onp.cos(noisy_phi))
    return z.ravel()

# ---- Fast costs ----
def fast_entropy(b):
    bp = b[b > 0]
    return float(-onp.dot(bp, onp.log(bp)))

def fast_rao(b):
    return float(b @ D_map_np @ b)

def map_belief_mean(b):
    return onp.einsum('k,klj->lj', b, all_maps)

print(f'Rollout pre-computation done.')
print(f'  Q_n: {Q_n_np.shape}, Y_n: {Y_n_np.shape}, codebook: {codebook_np.shape}')
print(f'  Hash entries: {len(_codebook_hash)}')

In [ ]:
# ----------------------------
# Run paired Monte Carlo comparison (common random numbers)
# ----------------------------
# ----------------------------
# Experiment configuration
# ----------------------------
seed = 10
rng = onp.random.default_rng(seed)

def sample_true_map_continuous(rng):
    """Sample a continuous true map m* in [0,10]x[0,10]."""
    xs = rng.uniform(0.0, 10.0, size=L_lm)
    ys = rng.uniform(0.0, 10.0, size=L_lm)
    return onp.stack([xs, ys], axis=1)

trial_specs = []
for _ in range(n_trials):
    trial_seed = int(rng.integers(0, 2**31 - 1))
    true_map = sample_true_map_continuous(rng)
    trial_specs.append((trial_seed, true_map))


## Experiment 2 – Visualization

Three panels:
- (a) Policy self-evaluation: each policy's stage cost under its own objective
- (b) Map estimation error: MSE against ground truth over time
- (c) Terminal error distribution: violin plot of final-step MSE


In [ ]:
# ----------------------------
# Richer rollout: per-timestep traces (CPU-optimized)
# ----------------------------

def rollout_traced(policy, trial_seed, true_map):
    """
    Returns per-timestep shannon_cost, rao_cost, mse.
    All-numpy, no GPU transfers.
    """
    local_rng = onp.random.default_rng(trial_seed)

    x = x0.copy()
    b = onp.full(len_M, 1.0 / len_M)

    shannon_trace = onp.empty(horizon)
    rao_trace = onp.empty(horizon)
    mse_trace = onp.empty(horizon)

    for t in range(horizon):
        # Record costs
        shannon_trace[t] = fast_entropy(b)
        rao_trace[t] = fast_rao(b)
        m_hat = map_belief_mean(b)
        mse_trace[t] = float(onp.mean((m_hat - true_map) ** 2))

        # Policy action
        b_idx = fast_belief_to_index(b)
        x_idx = fast_state_index(x)
        a_idx = int(policy[b_idx, x_idx])
        u = U_np[a_idx]

        # Step dynamics
        w = local_rng.multivariate_normal(onp.zeros(state_dim), cov_x)
        x_next = fast_step(x, u, w)

        # Observe
        v = onp.empty((L_lm, 2))
        v[:, 0] = local_rng.normal(0.0, _sigma_r, L_lm)
        v[:, 1] = local_rng.normal(0.0, _sigma_phi, L_lm)
        y = fast_observe(x_next, true_map, v)

        # Filter update
        x_idx_next = fast_state_index(x_next)
        b = fast_filter(b, x_idx_next, y)
        x = x_next

    return {
        'shannon': shannon_trace,
        'rao': rao_trace,
        'mse': mse_trace,
    }


def rollout_random(trial_seed, true_map):
    """Uniform random policy baseline — all numpy."""
    local_rng = onp.random.default_rng(trial_seed)

    x = x0.copy()
    b = onp.full(len_M, 1.0 / len_M)

    shannon_trace = onp.empty(horizon)
    rao_trace = onp.empty(horizon)
    mse_trace = onp.empty(horizon)

    for t in range(horizon):
        shannon_trace[t] = fast_entropy(b)
        rao_trace[t] = fast_rao(b)
        m_hat = map_belief_mean(b)
        mse_trace[t] = float(onp.mean((m_hat - true_map) ** 2))

        # Random action
        a_idx = local_rng.integers(0, U_np.shape[0])
        u = U_np[a_idx]


        w = local_rng.multivariate_normal(onp.zeros(state_dim), cov_x)
        x_next = fast_step(x, u, w)

        v = onp.empty((L_lm, 2))
        v[:, 0] = local_rng.normal(0.0, _sigma_r, L_lm)
        v[:, 1] = local_rng.normal(0.0, _sigma_phi, L_lm)
        y = fast_observe(x_next, true_map, v)

        x_idx_next = fast_state_index(x_next)
        b = fast_filter(b, x_idx_next, y)
        x = x_next

    return {
        'shannon': shannon_trace,
        'rao': rao_trace,
        'mse': mse_trace,
    }

In [ ]:
# ----------------------------
# Collect traces (common random numbers)
# ----------------------------
traces_shannon = []
traces_rao = []
traces_random = []

for trial_id, (trial_seed, true_map) in enumerate(
    tqdm(trial_specs, desc='Traced MC rollouts')
):
    traces_shannon.append(rollout_traced(vi_shannon['policy'], trial_seed, true_map))
    traces_rao.append(rollout_traced(vi_rao['policy'], trial_seed, true_map))
    traces_random.append(rollout_random(trial_seed, true_map))
# Stack into (n_trials, horizon) arrays
def stack_field(traces, field):
    return onp.stack([t[field] for t in traces], axis=0)  # (n_trials, horizon)

sh_shannon = stack_field(traces_shannon, 'shannon')  # shannon cost under shannon policy
ra_rao     = stack_field(traces_rao, 'rao')          # rao cost under rao policy
mse_sh     = stack_field(traces_shannon, 'mse')
mse_ra     = stack_field(traces_rao, 'mse')
mse_rand   = stack_field(traces_random, 'mse')

t_axis = onp.arange(horizon)

print(f'Traces collected: {n_trials} trials x {horizon} steps')


In [ ]:
# ----------------------------
# Plot
# ----------------------------
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 200,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})


def shaded_mean(ax, t, data, color, label, alpha_fill=0.15, band='ci95', linestyle='-', zorder=None):
    """Plot mean trajectory with uncertainty band (std or 95% CI)."""
    mu = data.mean(axis=0)
    sigma = data.std(axis=0)
    n = max(1, data.shape[0])

    if band == 'std':
        lo = mu - sigma
        hi = mu + sigma
    else:
        # Normal approx CI for mean trajectory
        half = 1.96 * sigma / onp.sqrt(n)
        lo = mu - half
        hi = mu + half

    kwargs = {'color': color, 'linewidth': 1.8, 'label': label, 'linestyle': linestyle}
    if zorder is not None:
        kwargs['zorder'] = zorder
    ax.plot(t, mu, **kwargs)
    ax.fill_between(t, lo, hi, color=color, alpha=alpha_fill, zorder=(zorder - 0.1) if zorder is not None else None)


fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

# ---- (a) Policy self-evaluation ----
ax = axes[0]
shaded_mean(ax, t_axis, sh_shannon, '#2980b9',
            r'$\mathbb{H}(b_t)$ under $\gamma_{\mathbb{H}}$')
shaded_mean(ax, t_axis, ra_rao, '#e67e22',
            r'$\tilde{W}_1(b_t)$ under $\gamma_{\tilde{W}}$')
ax.set_xlabel('Time step $t$')
ax.set_ylabel('Stage cost (own objective)')
ax.set_title('(a) Policy self-evaluation')
ax.legend(fontsize=9)

# ---- (b) MSE over time ---- (plot Shannon last + dashed so it's visible when overlapping Rao)
ax = axes[1]
shaded_mean(ax, t_axis, mse_rand, '#95a5a6', 'Random')
shaded_mean(ax, t_axis, mse_ra, '#e67e22',
            r'$\gamma_{\tilde{W}}$ (Rao)')
shaded_mean(ax, t_axis, mse_sh, '#2980b9',
            r'$\gamma_{\mathbb{H}}$ (Shannon)', linestyle='--', zorder=10)
ax.set_xlabel('Time step $t$')
ax.set_ylabel(r'MSE: $\mathbb{E}[\|m - \hat{m}\|^2]$')
ax.set_title('(b) Map estimation error')
ax.legend(fontsize=9)

# ---- (c) Terminal MSE distribution ----
ax = axes[2]
terminal_data = [mse_rand[:, -1], mse_sh[:, -1], mse_ra[:, -1]]
parts = ax.violinplot(terminal_data, positions=[1, 2, 3],
                      showmeans=True, showmedians=True)
colors_violin = ['#95a5a6', '#2980b9', '#e67e22']
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(colors_violin[i])
    pc.set_alpha(0.6)
parts['cmeans'].set_color('black')
parts['cmedians'].set_color('red')

# Overlay compact boxplots for quartiles to make tails easier to compare.
ax.boxplot(terminal_data, positions=[1, 2, 3], widths=0.15, patch_artist=False,
           showfliers=False, medianprops={'color': 'darkred'})

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['Random', r'$\gamma_{\mathbb{H}}$', r'$\gamma_{\tilde{W}}$'])
ax.set_ylabel('Terminal MSE')
ax.set_title('(c) Terminal error distribution (MC)')

fig.tight_layout()
import hashlib
n_landmarks = landmark_map.num_candidate_landmarks
lm_hash = hashlib.sha256(onp.round(to_numpy(landmark_map.cell_centers), 4).tobytes()).hexdigest()[:8]
exp2_fname = (
    f"exp2_n{pose_n}_nmap{map_n}_obs{obs_n}_act{action_n}_M{M}_beta{beta}_H{horizon}_T{n_trials}"
    f"_L{n_landmarks}_lm{lm_hash}_sw{sigma_w}_sr{sigma_r}_sp{sigma_phi}_eps{epsilon}_seed{seed}.png"
)
out_dir = PROJECT_ROOT / "output" / "experiment2"
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / exp2_fname, bbox_inches='tight')
plt.show()

# ----------------------------
# Summary statistics for paper
# ----------------------------
checkpoints = [0, max(0, horizon // 2), horizon - 1]
traj_summary = pd.DataFrame({
    't': checkpoints,
    'mse_random': [mse_rand[:, t].mean() for t in checkpoints],
    'mse_shannon': [mse_sh[:, t].mean() for t in checkpoints],
    'mse_rao': [mse_ra[:, t].mean() for t in checkpoints],
})
print('\n=== Mean MSE at key timesteps ===')
print(traj_summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

print('\n=== Terminal MSE Summary ===')
for name, data in [('Random', mse_rand[:, -1]),
                    ('Shannon', mse_sh[:, -1]),
                    ('Rao', mse_ra[:, -1])]:
    print(f'  {name:8s}: mean={data.mean():.4f}, std={data.std():.4f}, '
          f'median={onp.median(data):.4f}')

delta = mse_sh[:, -1] - mse_ra[:, -1]
print(f'\n  Paired delta (Shannon - Rao): mean={delta.mean():.4f}, '
      f'std={delta.std():.4f}')
print(f'  Fraction where Rao wins: {(delta > 0).mean():.2%}')


In [ ]:
# ----------------------------
# Transition-kernel diagnostics (coverage / stochasticity / branching)
# ----------------------------
# Requires updated src/classes/belief_mdp_n_M.py with transition_kernel_report().

report = bmdp.transition_kernel_report(row_sum_tolerance=1e-3, verbose=True)

# Per-action table
per_action_df = pd.DataFrame(report['per_action'])
cols = [
    'action_idx', 'row_coverage', 'nonempty_rows', 'empty_rows',
    'nnz', 'sparsity',
    'avg_nnz_all_rows', 'avg_nnz_nonempty_rows',
    'row_sum_min_nonempty', 'row_sum_mean_nonempty', 'row_sum_max_nonempty',
    'bad_row_count',
    'unique_next_belief_min', 'unique_next_belief_mean', 'unique_next_belief_max',
]
per_action_df = per_action_df[[c for c in cols if c in per_action_df.columns]].copy()

print('\nPer-action diagnostics:')
try:
    from IPython.display import display
    display(per_action_df)
except Exception:
    print(per_action_df.to_string(index=False))

# Aggregate table (single-row)
aggregate_row = {
    'problem_type': report['problem_type'],
    'cardinality': report['cardinality'],
    'm_n': report['m_n'],
    **report['aggregate'],
}
aggregate_df = pd.DataFrame([aggregate_row])

print('\nAggregate diagnostics:')
try:
    from IPython.display import display
    display(aggregate_df)
except Exception:
    print(aggregate_df.to_string(index=False))

# Optional: append aggregate rows across runs to compare kernels/settings.
# Example:
# diagnostics_log = []
# diagnostics_log.append({**aggregate_row, 'label': 'run_A'})
# pd.DataFrame(diagnostics_log)


In [ ]:
# ----------------------------
# MODEL-CONSISTENT sweep (rebuild planner per setting)
# Use this when varying sigma_r/sigma_phi/action_n/obs_n.
# This recomputes/loads T_mat, Q_n, p_n_M for each setting.
# ----------------------------
from scipy.stats import spearmanr
from datetime import datetime

CONSISTENT_BETA = [0.95, 0.99]
CONSISTENT_ACTION_N = [4, 8]
CONSISTENT_SIGMA_R = [0.50, 0.75]
CONSISTENT_SIGMA_PHI = [0.20, 0.35]
CONSISTENT_OBS_N = [obs_n]      # expand if desired
CONSISTENT_HORIZON = [20]       # rollout horizon can still vary separately
CONSISTENT_TRIALS = 60          # keep modest; planner rebuild is expensive
CONSISTENT_SEED = 6060

# Keep these from your current notebook config
POSE_N = pose_n
MAP_N = map_n
J_BATCH = 3876
I_BATCH = 1024
N_GPUS = n_gpus

stamp_consistent = datetime.now().strftime('%Y%m%d_%H%M%S')
consistent_dir = (
    PROJECT_ROOT / 'output' / 'experiment2' /
    f"sweep_model_consistent_M{M}_n{POSE_N}_nmap{MAP_N}_seed{CONSISTENT_SEED}_{stamp_consistent}"
)
consistent_dir.mkdir(parents=True, exist_ok=True)
print('Model-consistent output dir:', consistent_dir)


def _build_mapping_mdp(beta_local, action_n_local, obs_n_local, sigma_r_local, sigma_phi_local):
    sensor_local = RangeBearingSensor(
        r_max=float(sensor.r_max),
        epsilon=float(sensor.epsilon),
        sigma_r=float(sigma_r_local),
        sigma_phi=float(sigma_phi_local),
    )
    cov_y_local = onp.diag(onp.tile([sigma_r_local**2, sigma_phi_local**2], landmark_map.num_candidate_landmarks))

    mdp_local = BeliefMDP_n_M_Mapping(
        M=M,
        β=float(beta_local),
        n=POSE_N,
        motion_model=motion_model,
        measurement_model=sensor_local,
        obstacles=obstacles,
        _map=landmark_map,
        sigma_w=sigma_w,
        cov_y=cov_y_local,
        obs_n=int(obs_n_local),
        action_n=int(action_n_local),
        j_batch_size=J_BATCH,
        i_batch_size=I_BATCH,
        n_gpus=N_GPUS,
    )
    return mdp_local


def _cost_tensors_for_mdp(mdp_local):
    B_codebook_local = to_backend(mdp_local.BQ.Π_n_M)
    sh_local = to_backend(mdp_local.r_information_gain(B_codebook_local))
    ra_local = to_backend(mdp_local.rao_entropy(B_codebook_local))

    card_local = mdp_local.BQ.cardinality
    m_n_local = mdp_local.SQ.m_n
    n_u_local = mdp_local.AQ.n_u

    c_sh_2d = sh_local[:, np.newaxis]
    c_ra_2d = ra_local[:, np.newaxis]
    c_sh = np.broadcast_to(c_sh_2d[:, np.newaxis, :], (card_local, m_n_local, n_u_local)).copy()
    c_ra = np.broadcast_to(c_ra_2d[:, np.newaxis, :], (card_local, m_n_local, n_u_local)).copy()
    return c_sh, c_ra, sh_local, ra_local


def _run_vi_local(mdp_local, c_tensor, eps_local=1e-6):
    old_cost = mdp_local.c_n_M
    try:
        mdp_local.c_n_M = c_tensor
        vi_local = ValueIteration(mdp_local, epsilon=eps_local)
        vi_local.run(verbose=False)
        return to_numpy(vi_local.policy).astype(onp.int32), int(vi_local.iteration_count), to_numpy(vi_local.V)
    finally:
        mdp_local.c_n_M = old_cost


rows_consistent = []

for beta_local in CONSISTENT_BETA:
    for action_n_local in CONSISTENT_ACTION_N:
        for obs_n_local in CONSISTENT_OBS_N:
            for sr_local in CONSISTENT_SIGMA_R:
                for sp_local in CONSISTENT_SIGMA_PHI:
                    print(
                        f"\n[build] beta={beta_local}, action_n={action_n_local}, obs_n={obs_n_local}, "
                        f"sigma_r={sr_local}, sigma_phi={sp_local}"
                    )

                    mdp_local = _build_mapping_mdp(
                        beta_local=beta_local,
                        action_n_local=action_n_local,
                        obs_n_local=obs_n_local,
                        sigma_r_local=sr_local,
                        sigma_phi_local=sp_local,
                    )

                    # Kernel health diagnostics
                    rep = mdp_local.transition_kernel_report(verbose=False)
                    cov_mean = float(rep['aggregate']['row_coverage_mean'])
                    bad_rows = int(rep['aggregate']['bad_rows_total'])

                    # Costs + policy solve
                    c_sh, c_ra, sh_local, ra_local = _cost_tensors_for_mdp(mdp_local)
                    pol_sh, it_sh, V_sh = _run_vi_local(mdp_local, c_sh, eps_local=epsilon)
                    pol_ra, it_ra, V_ra = _run_vi_local(mdp_local, c_ra, eps_local=epsilon)

                    # Separation diagnostics
                    policy_diff_rate = float(onp.mean(pol_sh != pol_ra))
                    rho_spearman = float(spearmanr(to_numpy(sh_local), to_numpy(ra_local)).statistic)

                    # Optional lightweight terminal MSE test (same CRN)
                    # Reuse the currently active fast rollout utilities (from the earlier notebook cells)
                    # NOTE: This keeps simulator noise fixed at the local setting and horizon from CONSISTENT_HORIZON.
                    local_rng = onp.random.default_rng(CONSISTENT_SEED)
                    trial_specs_local = []
                    for _ in range(CONSISTENT_TRIALS):
                        tseed = int(local_rng.integers(0, 2**31 - 1))
                        tmap = sample_true_map_continuous(local_rng)
                        trial_specs_local.append((tseed, tmap))

                    # Evaluate with existing rollout functions (if they depend on global bmdp internals,
                    # skip MSE eval here and keep policy-level metrics)
                    mse_gap_mean = onp.nan
                    mse_gap_ci95 = onp.nan

                    try:
                        H_eval = int(CONSISTENT_HORIZON[0])
                        gaps = []
                        for tseed, tmap in trial_specs_local:
                            out_sh = _rollout_traced_param(pol_sh, tseed, tmap, H_eval, sr_local, sp_local)
                            out_ra = _rollout_traced_param(pol_ra, tseed, tmap, H_eval, sr_local, sp_local)
                            gaps.append(float(out_sh['mse'][-1] - out_ra['mse'][-1]))
                        gaps = onp.asarray(gaps, dtype=float)
                        mse_gap_mean = float(gaps.mean())
                        mse_gap_std = float(gaps.std(ddof=1)) if gaps.size > 1 else 0.0
                        mse_gap_ci95 = float(1.96 * mse_gap_std / max(1, onp.sqrt(gaps.size)))
                    except Exception as e:
                        print('  [note] terminal MSE eval skipped for this setting:', e)

                    row = {
                        'beta': beta_local,
                        'action_n': int(action_n_local),
                        'obs_n': int(obs_n_local),
                        'sigma_r': float(sr_local),
                        'sigma_phi': float(sp_local),
                        'policy_diff_rate': policy_diff_rate,
                        'cost_rank_spearman': rho_spearman,
                        'iter_shannon': int(it_sh),
                        'iter_rao': int(it_ra),
                        'kernel_row_coverage_mean': cov_mean,
                        'kernel_bad_rows_total': bad_rows,
                        'gap_mean_sh_minus_ra': mse_gap_mean,
                        'gap_ci95': mse_gap_ci95,
                    }
                    rows_consistent.append(row)

                    # Tiny per-setting summary image
                    fig, ax = plt.subplots(1, 1, figsize=(6.2, 3.5))
                    labels = ['policy_diff_rate', '1-rank_corr']
                    vals = [policy_diff_rate, max(0.0, 1.0 - abs(rho_spearman))]
                    ax.bar(labels, vals, color=['#2980b9', '#e67e22'])
                    ax.set_ylim(0, 1)
                    ax.set_title(
                        f"beta={beta_local}, act={action_n_local}, obs={obs_n_local}, "
                        f"sr={sr_local}, sp={sp_local}"
                    )
                    ax.set_ylabel('separation proxy')
                    fig.tight_layout()

                    img_name = (
                        f"consistent_sep_M{M}_n{POSE_N}_nmap{MAP_N}_obs{obs_n_local}_act{action_n_local}"
                        f"_beta{beta_local}_sr{sr_local}_sp{sp_local}_trials{CONSISTENT_TRIALS}_seed{CONSISTENT_SEED}.png"
                    )
                    fig.savefig(consistent_dir / img_name, bbox_inches='tight')
                    plt.close(fig)


consistent_df = pd.DataFrame(rows_consistent).sort_values(
    by=['policy_diff_rate', 'cost_rank_spearman'], ascending=[False, True]
).reset_index(drop=True)

csv_name = (
    f"consistent_summary_M{M}_n{POSE_N}_nmap{MAP_N}_obs{min(CONSISTENT_OBS_N)}-{max(CONSISTENT_OBS_N)}"
    f"_act{min(CONSISTENT_ACTION_N)}-{max(CONSISTENT_ACTION_N)}_seed{CONSISTENT_SEED}.csv"
)
consistent_df.to_csv(consistent_dir / csv_name, index=False)

print('\nModel-consistent sweep summary (top rows):')
try:
    from IPython.display import display
    display(consistent_df.head(20))
except Exception:
    print(consistent_df.head(20).to_string(index=False))

print('\nSaved model-consistent artifacts to:', consistent_dir)
print('  - per-setting separation images (.png)')
print('  - summary table (.csv)')

In [ ]:
# ----------------------------
# STREAMLINED MODEL-CONSISTENT SWEEP (same 3-panel plots per setting)
# ----------------------------
# Why previous run looked short: each setting can trigger a full p_n_M rebuild.
# This cell prints expected/finished setting counts and saves a full Experiment-2 style figure per setting.

from datetime import datetime

# ===== Sweep configuration =====
SWP_BETA = [0.95, 0.99]
SWP_ACTION_N = [4, 8]
SWP_OBS_N = [3]
SWP_SIGMA_R = [0.50, 0.75]
SWP_SIGMA_PHI = [0.20, 0.35]
SWP_HORIZON = 20
SWP_TRIALS = 80
SWP_SEED = 9090

# Safety controls
SWP_MAX_SETTINGS = None   # set int for debugging, else None
SWP_SKIP_IF_SUMMARY_EXISTS = False

# Planner compute controls
SWP_J_BATCH = 3876
SWP_I_BATCH = 1024
SWP_N_GPUS = n_gpus

stamp_swp = datetime.now().strftime('%Y%m%d_%H%M%S')
stream_dir = (
    PROJECT_ROOT / 'output' / 'experiment2' /
    f"sweep_streamlined_consistent_M{M}_n{pose_n}_nmap{map_n}_obs{min(SWP_OBS_N)}-{max(SWP_OBS_N)}_seed{SWP_SEED}_{stamp_swp}"
)
stream_dir.mkdir(parents=True, exist_ok=True)
print('Streamlined sweep dir:', stream_dir)

# Cartesian settings list
settings = []
for b in SWP_BETA:
    for an in SWP_ACTION_N:
        for on_ in SWP_OBS_N:
            for sr in SWP_SIGMA_R:
                for sp in SWP_SIGMA_PHI:
                    settings.append((b, an, on_, sr, sp))

if SWP_MAX_SETTINGS is not None:
    settings = settings[:int(SWP_MAX_SETTINGS)]

print(f'Planned settings: {len(settings)}')


def _build_mdp_local(beta_local, action_n_local, obs_n_local, sigma_r_local, sigma_phi_local):
    sensor_local = RangeBearingSensor(
        r_max=float(sensor.r_max),
        epsilon=float(sensor.epsilon),
        sigma_r=float(sigma_r_local),
        sigma_phi=float(sigma_phi_local),
    )
    cov_y_local = onp.diag(onp.tile([sigma_r_local**2, sigma_phi_local**2], landmark_map.num_candidate_landmarks))

    return BeliefMDP_n_M_Mapping(
        M=M,
        β=float(beta_local),
        n=pose_n,
        motion_model=motion_model,
        measurement_model=sensor_local,
        obstacles=obstacles,
        _map=landmark_map,
        sigma_w=sigma_w,
        cov_y=cov_y_local,
        obs_n=int(obs_n_local),
        action_n=int(action_n_local),
        j_batch_size=SWP_J_BATCH,
        i_batch_size=SWP_I_BATCH,
        n_gpus=SWP_N_GPUS,
    )


def _build_cost_tensors_local(mdp_local):
    B_codebook_local = to_backend(mdp_local.BQ.Π_n_M)
    sh_local = to_backend(mdp_local.r_information_gain(B_codebook_local))
    ra_local = to_backend(mdp_local.rao_entropy(B_codebook_local))

    card_local = mdp_local.BQ.cardinality
    m_n_local = mdp_local.SQ.m_n
    n_u_local = mdp_local.AQ.n_u

    c_sh_2d = sh_local[:, np.newaxis]
    c_ra_2d = ra_local[:, np.newaxis]
    c_sh = np.broadcast_to(c_sh_2d[:, np.newaxis, :], (card_local, m_n_local, n_u_local)).copy()
    c_ra = np.broadcast_to(c_ra_2d[:, np.newaxis, :], (card_local, m_n_local, n_u_local)).copy()
    return c_sh, c_ra, sh_local, ra_local


def _run_vi_local(mdp_local, c_tensor):
    old_cost = mdp_local.c_n_M
    try:
        mdp_local.c_n_M = c_tensor
        vi_local = ValueIteration(mdp_local, epsilon=epsilon)
        vi_local.run(verbose=False)
        return to_numpy(vi_local.policy).astype(onp.int32), int(vi_local.iteration_count), to_numpy(vi_local.V)
    finally:
        mdp_local.c_n_M = old_cost


def _make_local_rollout_backend(mdp_local, sigma_r_local, sigma_phi_local):
    """Create pure-numpy rollout closures tied to mdp_local (model-consistent)."""
    from scipy.spatial import KDTree as ScipyKDTree

    all_maps_local = to_numpy(mdp_local.all_maps_3d)
    Q_n_local = to_numpy(mdp_local.Q_n)
    Y_n_local = to_numpy(mdp_local.Y_n)
    X_n_local = to_numpy(mdp_local.SQ.X_n)
    U_local = to_numpy(mdp_local.AQ.U)
    codebook_local = to_numpy(mdp_local.BQ.Π_n_M)
    D_map_local = to_numpy(mdp_local._get_map_distance_matrix())

    L_local = all_maps_local.shape[1]
    len_M_local = all_maps_local.shape[0]
    state_dim_local = mdp_local.state_dim
    cov_x_local = to_numpy(mdp_local.cov_x)

    dt_local = float(mdp_local.motion_model.dt)
    lower_local = to_numpy(mdp_local.state_bounds[:, 0])
    upper_local = to_numpy(mdp_local.state_bounds[:, 1])
    eps_local = float(mdp_local.sensor.epsilon)
    rmax_local = float(mdp_local.sensor.r_max)

    Mq = int(mdp_local.BQ.M)
    codebook_int = onp.rint(codebook_local * Mq).astype(onp.int64)
    codebook_hash = {tuple(row): idx for idx, row in enumerate(codebook_int)}

    state_tree = ScipyKDTree(X_n_local)
    Y_n_nan = onp.isnan(Y_n_local)

    def reznik_quantize_local(b):
        k = onp.floor(Mq * b + 0.5).astype(onp.int64)
        delta = int(k.sum()) - Mq
        if delta != 0:
            d = k - Mq * b
            order = onp.argsort(d)
            if delta > 0:
                k[order[-delta:]] -= 1
            else:
                k[order[:-delta]] += 1
        return k

    def belief_to_index_local(b):
        k = reznik_quantize_local(b)
        idx = codebook_hash.get(tuple(k))
        if idx is not None:
            return idx
        q = k.astype(onp.float64) / Mq
        return int(onp.argmin(onp.linalg.norm(codebook_local - q, axis=1)))

    def state_index_local(x):
        return int(state_tree.query(x)[1])

    def obs_index_local(y_flat):
        nan_y = onp.isnan(y_flat)
        if onp.any(nan_y):
            matches = onp.all(Y_n_nan == nan_y, axis=1)
            if onp.any(matches):
                non_nan = ~nan_y
                d = onp.linalg.norm(Y_n_local[matches][:, non_nan] - y_flat[non_nan], axis=1)
                return int(onp.where(matches)[0][onp.argmin(d)])
        d = onp.linalg.norm(onp.nan_to_num(Y_n_local) - onp.nan_to_num(y_flat), axis=1)
        d = onp.where(Y_n_nan.any(axis=1), onp.inf, d)
        return int(onp.argmin(d))

    def fast_filter_local(b, x_idx, y_flat):
        oi = obs_index_local(y_flat)
        log_Q = onp.log(onp.maximum(Q_n_local[oi, x_idx, :], 1e-300))
        log_b = onp.log(onp.maximum(b, 1e-300))
        log_num = log_Q + log_b
        log_num -= log_num.max()
        b_new = onp.exp(log_num)
        s = b_new.sum()
        return b_new / s if s > 0 else onp.full(len_M_local, 1.0 / len_M_local)

    def step_local(x, u, w):
        return onp.clip(x + u * dt_local + w, lower_local, upper_local)

    def observe_local(x, true_map, v):
        delta = true_map - x[:2]
        ranges = onp.sqrt((delta**2).sum(axis=1))
        bearings = onp.arctan2(delta[:, 1], delta[:, 0])
        visible = (ranges >= eps_local) & (ranges <= rmax_local)
        z = onp.full((L_local, 2), onp.nan)
        if visible.any():
            z[visible, 0] = ranges[visible] + v[visible, 0]
            noisy_phi = bearings[visible] + v[visible, 1]
            z[visible, 1] = onp.arctan2(onp.sin(noisy_phi), onp.cos(noisy_phi))
        return z.ravel()

    def entropy_local(b):
        bp = b[b > 0]
        return float(-onp.dot(bp, onp.log(bp)))

    def rao_local(b):
        return float(b @ D_map_local @ b)

    def map_mean_local(b):
        return onp.einsum('k,klj->lj', b, all_maps_local)

    def rollout_policy(policy, trial_seed, true_map, horizon_local):
        rng = onp.random.default_rng(trial_seed)
        x = onp.array([5.0, 5.0], dtype=float)
        b = onp.full(len_M_local, 1.0 / len_M_local)

        sh = onp.empty(horizon_local)
        ra = onp.empty(horizon_local)
        mse = onp.empty(horizon_local)

        for t in range(horizon_local):
            sh[t] = entropy_local(b)
            ra[t] = rao_local(b)
            mse[t] = float(onp.mean((map_mean_local(b) - true_map) ** 2))

            bi = belief_to_index_local(b)
            xi = state_index_local(x)
            ai = int(policy[bi, xi])
            u = U_local[ai]

            w = rng.multivariate_normal(onp.zeros(state_dim_local), cov_x_local)
            x_next = step_local(x, u, w)

            v = onp.empty((L_local, 2))
            v[:, 0] = rng.normal(0.0, sigma_r_local, L_local)
            v[:, 1] = rng.normal(0.0, sigma_phi_local, L_local)
            y = observe_local(x_next, true_map, v)

            x_idx_next = state_index_local(x_next)
            b = fast_filter_local(b, x_idx_next, y)
            x = x_next

        return {'shannon': sh, 'rao': ra, 'mse': mse}

    def rollout_random(trial_seed, true_map, horizon_local):
        rng = onp.random.default_rng(trial_seed)
        x = onp.array([5.0, 5.0], dtype=float)
        b = onp.full(len_M_local, 1.0 / len_M_local)

        sh = onp.empty(horizon_local)
        ra = onp.empty(horizon_local)
        mse = onp.empty(horizon_local)

        for t in range(horizon_local):
            sh[t] = entropy_local(b)
            ra[t] = rao_local(b)
            mse[t] = float(onp.mean((map_mean_local(b) - true_map) ** 2))

            ai = int(rng.integers(0, U_local.shape[0]))
            u = U_local[ai]

            w = rng.multivariate_normal(onp.zeros(state_dim_local), cov_x_local)
            x_next = step_local(x, u, w)

            v = onp.empty((L_local, 2))
            v[:, 0] = rng.normal(0.0, sigma_r_local, L_local)
            v[:, 1] = rng.normal(0.0, sigma_phi_local, L_local)
            y = observe_local(x_next, true_map, v)

            x_idx_next = state_index_local(x_next)
            b = fast_filter_local(b, x_idx_next, y)
            x = x_next

        return {'shannon': sh, 'rao': ra, 'mse': mse}

    return rollout_policy, rollout_random


def _make_trial_specs_local(n_trials_local, seed_local):
    rng = onp.random.default_rng(seed_local)
    specs = []
    for _ in range(n_trials_local):
        tseed = int(rng.integers(0, 2**31 - 1))
        tmap = sample_true_map_continuous(rng)
        specs.append((tseed, tmap))
    return specs


rows = []
completed = 0

for idx, (beta_local, action_n_local, obs_n_local, sr_local, sp_local) in enumerate(settings, start=1):
    print(
        f"\n[{idx}/{len(settings)}] build beta={beta_local}, action_n={action_n_local}, "
        f"obs_n={obs_n_local}, sigma_r={sr_local}, sigma_phi={sp_local}"
    )

    mdp_local = _build_mdp_local(beta_local, action_n_local, obs_n_local, sr_local, sp_local)
    rep = mdp_local.transition_kernel_report(verbose=False)

    c_sh, c_ra, sh_local, ra_local = _build_cost_tensors_local(mdp_local)
    pol_sh, it_sh, _ = _run_vi_local(mdp_local, c_sh)
    pol_ra, it_ra, _ = _run_vi_local(mdp_local, c_ra)

    policy_diff_rate = float(onp.mean(pol_sh != pol_ra))
    rank_corr = float(spearmanr(to_numpy(sh_local), to_numpy(ra_local)).statistic)

    rollout_policy_local, rollout_random_local = _make_local_rollout_backend(mdp_local, sr_local, sp_local)
    specs = _make_trial_specs_local(SWP_TRIALS, SWP_SEED)

    tr_sh, tr_ra, tr_rand = [], [], []
    for tseed, tmap in specs:
        tr_sh.append(rollout_policy_local(pol_sh, tseed, tmap, SWP_HORIZON))
        tr_ra.append(rollout_policy_local(pol_ra, tseed, tmap, SWP_HORIZON))
        tr_rand.append(rollout_random_local(tseed, tmap, SWP_HORIZON))

    def stack_field(traces, field):
        return onp.stack([t[field] for t in traces], axis=0)

    sh_shannon = stack_field(tr_sh, 'shannon')
    ra_rao = stack_field(tr_ra, 'rao')
    mse_sh = stack_field(tr_sh, 'mse')
    mse_ra = stack_field(tr_ra, 'mse')
    mse_rand = stack_field(tr_rand, 'mse')

    delta = mse_sh[:, -1] - mse_ra[:, -1]
    gap_mean = float(delta.mean())
    gap_std = float(delta.std(ddof=1)) if delta.size > 1 else 0.0
    gap_ci95 = float(1.96 * gap_std / max(1, onp.sqrt(delta.size)))

    # --- SAME 3-panel style plot as before ---
    t_axis = onp.arange(SWP_HORIZON)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

    def shaded_mean(ax, t, data, color, label, alpha_fill=0.15):
        mu = data.mean(axis=0)
        sigma = data.std(axis=0)
        n = max(1, data.shape[0])
        half = 1.96 * sigma / onp.sqrt(n)
        ax.plot(t, mu, color=color, linewidth=1.8, label=label)
        ax.fill_between(t, mu - half, mu + half, color=color, alpha=alpha_fill)

    # (a) self-evaluation
    shaded_mean(axes[0], t_axis, sh_shannon, '#2980b9', r'$\\mathbb{H}(b_t)$ under $\\gamma_{\\mathbb{H}}$')
    shaded_mean(axes[0], t_axis, ra_rao, '#e67e22', r'$\\tilde{W}_1(b_t)$ under $\\gamma_{\\tilde{W}}$')
    axes[0].set_title('(a) Policy self-evaluation')
    axes[0].set_xlabel('Time step $t$')
    axes[0].set_ylabel('Stage cost (own objective)')
    axes[0].legend(fontsize=9)

    # (b) MSE over time
    shaded_mean(axes[1], t_axis, mse_rand, '#95a5a6', 'Random')
    shaded_mean(axes[1], t_axis, mse_ra, '#e67e22', r'$\\gamma_{\\tilde{W}}$ (Rao)')
    shaded_mean(axes[1], t_axis, mse_sh, '#2980b9', r'$\\gamma_{\\mathbb{H}}$ (Shannon)')
    axes[1].set_title('(b) Map estimation error')
    axes[1].set_xlabel('Time step $t$')
    axes[1].set_ylabel('MSE')
    axes[1].legend(fontsize=9)

    # (c) terminal MSE distribution
    terminal_data = [mse_rand[:, -1], mse_sh[:, -1], mse_ra[:, -1]]
    parts = axes[2].violinplot(terminal_data, positions=[1, 2, 3], showmeans=True, showmedians=True)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(['#95a5a6', '#2980b9', '#e67e22'][i])
        pc.set_alpha(0.6)
    parts['cmeans'].set_color('black')
    parts['cmedians'].set_color('red')
    axes[2].boxplot(terminal_data, positions=[1, 2, 3], widths=0.15, patch_artist=False, showfliers=False)
    axes[2].set_xticks([1, 2, 3])
    axes[2].set_xticklabels(['Random', r'$\\gamma_{\\mathbb{H}}$', r'$\\gamma_{\\tilde{W}}$'])
    axes[2].set_title('(c) Terminal error distribution (MC)')
    axes[2].set_ylabel('Terminal MSE')

    fig.suptitle(
        f"beta={beta_local}, act={action_n_local}, obs={obs_n_local}, sr={sr_local}, sp={sp_local}, "
        f"policy_diff={100*policy_diff_rate:.2f}%"
    )
    fig.tight_layout()

    img_name = (
        f"sameplot_M{M}_n{pose_n}_nmap{map_n}_obs{obs_n_local}_act{action_n_local}"
        f"_beta{beta_local}_H{SWP_HORIZON}_sr{sr_local}_sp{sp_local}"
        f"_trials{SWP_TRIALS}_seed{SWP_SEED}.png"
    )
    fig.savefig(stream_dir / img_name, bbox_inches='tight')
    plt.close(fig)

    rows.append({
        'beta': beta_local,
        'action_n': int(action_n_local),
        'obs_n': int(obs_n_local),
        'sigma_r': float(sr_local),
        'sigma_phi': float(sp_local),
        'iter_shannon': int(it_sh),
        'iter_rao': int(it_ra),
        'policy_diff_rate': policy_diff_rate,
        'cost_rank_spearman': rank_corr,
        'kernel_row_coverage_mean': float(rep['aggregate']['row_coverage_mean']),
        'kernel_bad_rows_total': int(rep['aggregate']['bad_rows_total']),
        'final_gap_mean_sh_minus_ra': gap_mean,
        'final_gap_ci95': gap_ci95,
        'rao_win_rate': float((delta > 0).mean()),
    })
    completed += 1
    print(f"  completed {completed}/{len(settings)} | gap={gap_mean:.4f} +/- {gap_ci95:.4f}")


summary_df = pd.DataFrame(rows).sort_values(
    by=['policy_diff_rate', 'final_gap_mean_sh_minus_ra'],
    ascending=[False, False]
).reset_index(drop=True)

summary_name = (
    f"streamlined_summary_M{M}_n{pose_n}_nmap{map_n}_obs{min(SWP_OBS_N)}-{max(SWP_OBS_N)}"
    f"_act{min(SWP_ACTION_N)}-{max(SWP_ACTION_N)}_trials{SWP_TRIALS}_seed{SWP_SEED}.csv"
)
summary_df.to_csv(stream_dir / summary_name, index=False)

print(f"\nFinished settings: {completed}/{len(settings)}")
print('Saved to:', stream_dir)
try:
    from IPython.display import display
    display(summary_df.head(20))
except Exception:
    print(summary_df.head(20).to_string(index=False))
